#### DUPLICITY PROFILE


##### 06.1 DOCUMENTAÇÃO DO DUPLICITY PROFILE

###### Objetivo

O **Duplicity Profile** tem como objetivo identificar, quantificar e analisar a ocorrência de duplicidades nos registros e atributos do DataFrame.

Enquanto o **Schema Profile** analisa a estrutura das colunas, o **Distribution Profile** analisa a distribuição dos valores e o **Pattern Profile** identifica padrões estruturais, o **Duplicity Profile** avalia a repetição de informações e o potencial de unicidade dos dados.

A análise não depende do significado de negócio das colunas.

O foco está nas características observáveis dos dados, como:

- duplicidade completa de registros;
- grupos de registros idênticos;
- quantidade de registros envolvidos em duplicações;
- duplicações excedentes;
- frequência de repetição;
- unicidade por coluna;
- valores distintos e repetidos;
- possíveis chaves candidatas;
- possíveis chaves compostas;
- concentração dos grupos duplicados.

###### Exemplos

Considere os registros:

| ID | Produto | Quantidade |
|---|---|---|
| 101 | Produto A | 2 |
| 101 | Produto A | 2 |
| 102 | Produto B | 3 |
| 103 | Produto C | 1 |
| 103 | Produto C | 1 |
| 103 | Produto C | 1 |

Nesse conjunto existem:

- 6 registros no total;
- 3 combinações distintas de registros;
- 2 grupos com duplicidade;
- 5 registros envolvidos em grupos duplicados;
- 3 registros duplicados excedentes.

A duplicidade excedente corresponde à quantidade de registros que ultrapassa uma ocorrência por grupo.

Neste exemplo:

`Duplicações excedentes = 6 - 3 = 3`

O percentual de duplicidade excedente é:

`(3 / 6) × 100 = 50%`

Já o percentual de registros envolvidos em grupos duplicados é:

`(5 / 6) × 100 = 83,33%`

Esses indicadores representam perspectivas diferentes da duplicidade.

###### Principais análises

- Identificação de registros completamente duplicados;
- Contagem de combinações distintas de registros;
- Quantidade de grupos com duplicidade;
- Quantidade de registros envolvidos em duplicações;
- Quantidade e percentual de duplicações excedentes;
- Frequência dos grupos duplicados;
- Ranking dos grupos com maior repetição;
- Quantidade de valores distintos por coluna;
- Percentual de unicidade por coluna;
- Identificação de possíveis chaves candidatas simples;
- Avaliação controlada de possíveis chaves compostas;
- Consolidação dos indicadores de duplicidade;
- Resumo executivo do Duplicity Profile.

###### Interpretação dos resultados

A ocorrência de valores repetidos não representa necessariamente um problema de qualidade dos dados.

Uma coluna que representa município, categoria ou situação operacional pode apresentar repetições legítimas.

Da mesma forma, registros completamente idênticos podem ocorrer em determinadas fontes sem representar necessariamente um erro.

Por esse motivo, o **Duplicity Profile** diferencia:

- **Repetição de valores:** ocorrência de um mesmo valor em uma coluna;
- **Duplicidade completa:** ocorrência de registros idênticos considerando todas as colunas analisadas;
- **Registros envolvidos em duplicações:** total de registros pertencentes a grupos com mais de uma ocorrência;
- **Duplicações excedentes:** quantidade de ocorrências adicionais à primeira de cada grupo;
- **Unicidade:** capacidade de uma coluna ou combinação de colunas apresentar valores exclusivos;
- **Chave candidata:** atributo ou combinação de atributos com potencial para identificar registros de forma única.

As chaves identificadas são consideradas candidatas estatísticas e não são automaticamente classificadas como chaves de negócio.

A confirmação de uma regra de unicidade depende do contexto e dos requisitos da fonte de dados.

###### Característica

O **Duplicity Profile** é independente da origem dos dados.

Sua análise é realizada exclusivamente sobre o DataFrame disponibilizado pelo **Data Preparation**.

O processamento utiliza operações distribuídas do Apache Spark para analisar registros, valores distintos e combinações de atributos.

A avaliação de chaves compostas utiliza limites configuráveis para evitar a exploração indiscriminada de combinações de colunas e reduzir o custo computacional.

As funções genéricas e reutilizáveis são mantidas no notebook de bibliotecas **00_FRAMEWORK_CONFIG**, enquanto o **Duplicity Profile** concentra as configurações específicas, análises, consolidações e visualizações.

Dessa forma, o mesmo Profile pode ser utilizado em diferentes fontes, volumes e estruturas de dados.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- total de registros analisados;
- quantidade de registros distintos;
- quantidade de grupos duplicados;
- quantidade de registros envolvidos em duplicações;
- quantidade e percentual de duplicações excedentes;
- ranking dos grupos duplicados mais frequentes;
- indicadores de unicidade por coluna;
- identificação de possíveis chaves candidatas simples;
- identificação de possíveis chaves compostas;
- métricas consolidadas de duplicidade;
- resumo executivo dos resultados;
- DataFrame consolidado do Duplicity Profile.

Os principais DataFrames disponibilizados são:

- `duplicity_profile_df`
- `duplicity_summary_df`
- `duplicity_full_groups_df`
- `duplicity_top_groups_df`
- `duplicity_composite_keys_df`

Esses resultados poderão ser utilizados posteriormente pelo **Data Quality Score**, permitindo incorporar evidências de duplicidade e unicidade à avaliação geral da qualidade dos dados, respeitando a aplicabilidade de cada métrica.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 06.3 CONFIGURAÇÕES
# ============================================================

# O QUE FAZ: 
# Define limites e política de valores.

# COMO FAZ: 
# Utiliza parâmetros locais do Profile.

# POR QUE É IMPORTANTE: 
# Permite controlar custo e interpretação.

# PERGUNTA RESPONDIDA: 
# "Quais regras e limites serão aplicados?"

DUPLICITY_TOP_N = 5
DUPLICITY_MAX_CANDIDATE_COLUMNS = 8
DUPLICITY_MAX_COMPOSITE_PAIRS = 15
DUPLICITY_ENABLE_COMPOSITE = True
DUPLICITY_ENABLE_FULL = True
DUPLICITY_IGNORE_BLANK_STRINGS = True
DUPLICITY_SHOW_RESULTS = True
DUPLICITY_SHOW_DETAILS = False
DUPLICITY_RUN_TESTS = False
DUPLICITY_COLUMN_BATCH_SIZE = 4
DUPLICITY_PAIR_BATCH_SIZE = 3
# A seleção de pares é heurística; não garante encontrar todas as chaves compostas.

In [0]:
# ============================================================
# 06.4 VALIDAÇÃO E ELEGIBILIDADE
# ============================================================

# O QUE FAZ:
# Valida a existência dos dados preparados e identifica tipos elegíveis.

# COMO FAZ:
# Verifica schema, parâmetros, contagem de registros e funções compartilhadas.

# POR QUE É IMPORTANTE:
# Evita cálculos incorretos e falhas de execução tardias.

# PERGUNTA RESPONDIDA:
# "O DataFrame e suas colunas podem ser analisados?"

if "df_prepared" not in globals() or not isinstance(df_prepared, DataFrame):
    raise ValueError("df_prepared deve ser um DataFrame PySpark.")
if "total_registros" not in globals() or not isinstance(total_registros, int) or total_registros <= 0:
    raise ValueError("total_registros deve ser inteiro positivo.")
if not df_prepared.columns or len(df_prepared.columns) != len(set(df_prepared.columns)):
    raise ValueError("O DataFrame deve conter colunas com nomes únicos.")
for funcao in ("dq_col", "dq_valid_value", "dq_is_atomic", "dq_duplicate_group_metrics"):
    if funcao not in globals() or not callable(globals()[funcao]):
        raise RuntimeError(f"Função ausente no Notebook 00: {funcao}")
for nome, valor in (
    ("DUPLICITY_TOP_N", DUPLICITY_TOP_N),
    ("DUPLICITY_MAX_CANDIDATE_COLUMNS", DUPLICITY_MAX_CANDIDATE_COLUMNS),
    ("DUPLICITY_MAX_COMPOSITE_PAIRS", DUPLICITY_MAX_COMPOSITE_PAIRS),
    ("DUPLICITY_COLUMN_BATCH_SIZE", DUPLICITY_COLUMN_BATCH_SIZE),
    ("DUPLICITY_PAIR_BATCH_SIZE", DUPLICITY_PAIR_BATCH_SIZE),
):
    if not isinstance(valor, int) or isinstance(valor, bool) or valor <= 0:
        raise ValueError(f"{nome} deve ser inteiro positivo.")
campos = df_prepared.schema.fields
tipos = {campo.name: campo.dataType for campo in campos}
atomicas = [campo.name for campo in campos if dq_is_atomic(campo.dataType)]
complexas = [campo.name for campo in campos if not dq_is_atomic(campo.dataType)]
print(f"Registros: {total_registros:,} | Colunas: {len(campos)} | Elegíveis: {len(atomicas)}")

In [0]:
# ============================================================
# 06.5 DUPLICIDADE COMPLETA
# ============================================================

# O QUE FAZ:
# Quantifica a repetição de registros considerando todas as colunas.

# COMO FAZ:
# Agrupa as colunas elegíveis, consolida frequências e calcula métricas de grupos.

# POR QUE É IMPORTANTE:
# Distingue duplicações excedentes de registros envolvidos em duplicações.

# PERGUNTA RESPONDIDA:
# "Quantos registros completos estão duplicados?"

full_schema = """status string, total_registros long, grupos_distintos long,
grupos_duplicados long, registros_em_grupos_duplicados long,
linhas_duplicadas_excedentes long, maior_grupo long,
pct_linhas_duplicadas_excedentes double, pct_registros_em_grupos_duplicados double"""

duplicity_full_groups_df = None
duplicity_top_groups_df = None
if not DUPLICITY_ENABLE_FULL:
    duplicity_full_status = "NAO_EXECUTADO_CONFIGURACAO"
elif complexas:
    duplicity_full_status = "NAO_EXECUTADO_TIPOS_COMPLEXOS"
else:
    duplicity_full_status = "EXECUTADO"

metric_names = (
    "grupos_distintos", "grupos_duplicados", "registros_em_grupos_duplicados",
    "linhas_duplicadas_excedentes", "maior_grupo"
)
resumo_linhas = {nome: None for nome in metric_names}
if duplicity_full_status == "EXECUTADO":
    duplicity_full_groups_df = df_prepared.groupBy(
        *[dq_col(nome) for nome in atomicas]
    ).agg(F.count(F.lit(1)).alias("ocorrencias"))
    resumo_linhas = dq_duplicate_group_metrics(duplicity_full_groups_df).first().asDict()
    duplicity_top_groups_df = (
        duplicity_full_groups_df.filter(F.col("ocorrencias") > 1)
        .orderBy(F.desc("ocorrencias"))
        .limit(DUPLICITY_TOP_N)
    )
resumo_full = (
    duplicity_full_status, int(total_registros),
    *[int(resumo_linhas[nome]) if resumo_linhas[nome] is not None else None
      for nome in metric_names],
    (100.0 * resumo_linhas["linhas_duplicadas_excedentes"] / total_registros
     if resumo_linhas["linhas_duplicadas_excedentes"] is not None else None),
    (100.0 * resumo_linhas["registros_em_grupos_duplicados"] / total_registros
     if resumo_linhas["registros_em_grupos_duplicados"] is not None else None)
)
duplicity_full_summary_df = spark.createDataFrame([resumo_full], full_schema)
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_full_summary_df)
    if DUPLICITY_SHOW_DETAILS and duplicity_top_groups_df is not None:
        display(duplicity_top_groups_df)

In [0]:
# ============================================================
# 06.6 UNICIDADE POR COLUNA
# ============================================================

# O QUE FAZ:
# Calcula preenchimento, distintos válidos e repetição excedente por atributo.

# COMO FAZ:
# Executa agregações exatas em lotes de colunas e reúne métricas no driver.

# POR QUE É IMPORTANTE:
# Reduz o tamanho de cada plano Spark e identifica potenciais chaves simples.

# PERGUNTA RESPONDIDA:
# "Quais colunas são únicas e totalmente preenchidas?"

metricas_por_indice = {}
colunas_elegiveis = [(i, campo) for i, campo in enumerate(campos)
                     if dq_is_atomic(campo.dataType)]
for inicio in range(0, len(colunas_elegiveis), DUPLICITY_COLUMN_BATCH_SIZE):
    lote = colunas_elegiveis[inicio:inicio + DUPLICITY_COLUMN_BATCH_SIZE]
    expressoes = []
    for i, campo in lote:
        referencia = dq_col(campo.name)
        valido = dq_valid_value(
            campo.name, campo.dataType, DUPLICITY_IGNORE_BLANK_STRINGS
        )
        expressoes.extend([
            F.sum(F.when(referencia.isNull(), 1).otherwise(0)).alias(f"m{i}_null"),
            F.sum(F.when(referencia.isNotNull() & (~valido), 1).otherwise(0))
             .alias(f"m{i}_invalidos_nao_null"),
            F.sum(F.when(valido, 1).otherwise(0)).alias(f"m{i}_validos"),
            F.countDistinct(F.when(valido, referencia)).alias(f"m{i}_distintos")
        ])
    resultado_lote = df_prepared.agg(*expressoes).first().asDict()
    metricas_por_indice.update(resultado_lote)

resultados_colunas = []
for i, campo in enumerate(campos):
    if not dq_is_atomic(campo.dataType):
        resultados_colunas.append((
            campo.name, campo.dataType.simpleString(), "NAO_ELEGIVEL_TIPO_COMPLEXO",
            None, None, None, None, None, None, False
        ))
        continue
    nulos = int(metricas_por_indice[f"m{i}_null"] or 0)
    invalidos = int(metricas_por_indice[f"m{i}_invalidos_nao_null"] or 0)
    validos = int(metricas_por_indice[f"m{i}_validos"] or 0)
    distintos = int(metricas_por_indice[f"m{i}_distintos"] or 0)
    chave = validos == total_registros and distintos == total_registros
    resultados_colunas.append((
        campo.name, campo.dataType.simpleString(), "EXECUTADO", nulos,
        invalidos, validos, distintos, validos - distintos,
        100.0 * distintos / validos if validos else None, chave
    ))

schema_colunas = """coluna string, tipo_spark string, status string, null long,
invalidos_nao_null long, valores_validos long, distintos_validos long,
repeticoes_excedentes long, pct_unicidade_validos double, chave_candidata_simples boolean"""
duplicity_column_profile_df = spark.createDataFrame(resultados_colunas, schema_colunas)
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_column_profile_df.orderBy(F.desc("pct_unicidade_validos")))

In [0]:
# ============================================================
# 06.7 CHAVES COMPOSTAS CANDIDATAS
# ============================================================

# O QUE FAZ:
# Avalia pares de colunas quanto à capacidade de identificar registros.

# COMO FAZ:
# Seleciona candidatos heurísticos e calcula cardinalidade exata em lotes.

# POR QUE É IMPORTANTE:
# Controla a explosão combinatória e explicita que a busca não é exaustiva.

# PERGUNTA RESPONDIDA:
# "Quais pares de colunas podem identificar registros unicamente?"

nomes_candidatos = []
pares = []
if DUPLICITY_ENABLE_COMPOSITE and DUPLICITY_MAX_CANDIDATE_COLUMNS >= 2:
    selecionadas = (
        duplicity_column_profile_df
        .filter(
            (F.col("status") == "EXECUTADO") &
            (F.col("valores_validos") > 0) &
            (~F.col("chave_candidata_simples"))
        )
        .orderBy(F.desc("pct_unicidade_validos"), F.asc("coluna"))
        .limit(DUPLICITY_MAX_CANDIDATE_COLUMNS)
        .select("coluna").collect()
    )
    nomes_candidatos = [linha["coluna"] for linha in selecionadas]
    pares = list(combinations(nomes_candidatos, 2))[:DUPLICITY_MAX_COMPOSITE_PAIRS]

metricas_pares = {}
for inicio in range(0, len(pares), DUPLICITY_PAIR_BATCH_SIZE):
    lote = list(enumerate(pares))[inicio:inicio + DUPLICITY_PAIR_BATCH_SIZE]
    expressoes = []
    for i, (a, b) in lote:
        valido = (
            dq_valid_value(a, tipos[a], DUPLICITY_IGNORE_BLANK_STRINGS) &
            dq_valid_value(b, tipos[b], DUPLICITY_IGNORE_BLANK_STRINGS)
        )
        expressoes.extend([
            F.sum(F.when(valido, 1).otherwise(0)).alias(f"p{i}_validos"),
            F.countDistinct(
                F.when(valido, F.struct(dq_col(a).alias("a"), dq_col(b).alias("b")))
            ).alias(f"p{i}_distintos")
        ])
    metricas_pares.update(df_prepared.agg(*expressoes).first().asDict())

pair_results = []
for i, (a, b) in enumerate(pares):
    validos = int(metricas_pares[f"p{i}_validos"] or 0)
    distintos = int(metricas_pares[f"p{i}_distintos"] or 0)
    pair_results.append((
        a, b, validos, distintos, validos - distintos,
        100.0 * distintos / validos if validos else None,
        validos == total_registros and distintos == total_registros
    ))

pair_schema = """coluna_a string, coluna_b string, pares_validos long,
pares_distintos long, repeticoes_excedentes long,
pct_unicidade_pares_validos double, chave_candidata_composta boolean"""
duplicity_composite_keys_df = spark.createDataFrame(pair_results, pair_schema)
# Cobertura da busca de pares: apenas combinações entre colunas selecionadas.
qtd_pares_possiveis = len(nomes_candidatos) * (len(nomes_candidatos) - 1) // 2
print(f"Pares avaliados: {len(pares)} de {qtd_pares_possiveis} possíveis entre candidatos selecionados.")
if DUPLICITY_SHOW_RESULTS and DUPLICITY_SHOW_DETAILS:
    display(duplicity_composite_keys_df.orderBy(F.desc("pct_unicidade_pares_validos")))

In [0]:
# ============================================================
# 06.8 CONSOLIDAÇÃO E RESUMO EXECUTIVO
# ============================================================

# O QUE FAZ:
# Publica os indicadores consolidados de linhas, colunas e pares.

# COMO FAZ:
# Reutiliza DataFrames existentes e acrescenta contagens de chaves e cobertura.

# POR QUE É IMPORTANTE:
# Oferece contratos consistentes para o Data Quality Score e a observabilidade.

# PERGUNTA RESPONDIDA:
# "Quais evidências de duplicidade e unicidade foram produzidas?"

duplicity_profile_df = duplicity_column_profile_df
qtd_chaves_simples = sum(1 for linha in resultados_colunas if linha[-1])
qtd_chaves_compostas = sum(1 for linha in pair_results if linha[-1])
duplicity_summary_df = (
    duplicity_full_summary_df
    .withColumn("colunas_chave_simples", F.lit(qtd_chaves_simples))
    .withColumn("pares_avaliados", F.lit(len(pares)))
    .withColumn("pares_chave_composta", F.lit(qtd_chaves_compostas))
    .withColumn("colunas_nao_elegiveis", F.lit(len(complexas)))
    .withColumn("colunas_elegiveis", F.lit(len(atomicas)))
    .withColumn("pares_possiveis_candidatos", F.lit(qtd_pares_possiveis))
    .withColumn("busca_pares_exaustiva", F.lit(False))
    .withColumn("analise_completa_executada", F.lit(duplicity_full_status == "EXECUTADO"))
)
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_summary_df)
print("Saídas: duplicity_profile_df, duplicity_summary_df, duplicity_full_groups_df,")
print("duplicity_top_groups_df, duplicity_composite_keys_df")

In [0]:
# ============================================================
# 06.9 TESTES OPCIONAIS DE SANIDADE
# ============================================================

# O QUE FAZ:
# Valida invariantes matemáticas com dados sintéticos conhecidos.

# COMO FAZ:
# Executa testes de duplicidade e política de strings somente quando habilitados.

# POR QUE É IMPORTANTE:
# Detecta regressões sem impor ações extras a cada execução produtiva.

# PERGUNTA RESPONDIDA:
# "Os indicadores reproduzem resultados conhecidos?"

if DUPLICITY_RUN_TESTS:
    teste = spark.createDataFrame([
        ("A", 1, "x"), ("A", 1, "x"), ("A", 1, "y"),
        ("B", 2, "z"), ("B", 2, "z"), ("C", 3, "w")
    ], ["codigo", "valor", "rotulo"])
    grupos_teste = (
        teste.groupBy("codigo", "valor", "rotulo")
        .count().withColumnRenamed("count", "ocorrencias")
    )
    t = dq_duplicate_group_metrics(grupos_teste).first()
    assert t["grupos_distintos"] == 4
    assert t["grupos_duplicados"] == 2
    assert t["registros_em_grupos_duplicados"] == 4
    assert t["linhas_duplicadas_excedentes"] == 2
    assert t["linhas_duplicadas_excedentes"] == 6 - t["grupos_distintos"]
    strings = spark.createDataFrame([("A",), ("",), ("   ",), (None,)], ["codigo"])
    assert strings.filter(dq_valid_value("codigo", StringType(), True)).count() == 1
    assert strings.filter(dq_valid_value("codigo", StringType(), False)).count() == 3
    assert teste.select("codigo", "rotulo").distinct().count() == 4
    assert duplicity_column_profile_df.count() == len(campos)
    print("Testes sintéticos do Duplicity Profile: OK")